In [18]:
# ============================================================
# 🧰 Install dependencies + clone FineWeb2 repo
# ============================================================
!pip install git+https://github.com/huggingface/datatrove.git -q
!pip install pandas matplotlib tqdm ftfy trafilatura pyyaml fasteners fasttext-numpy2-wheel orjson -q
!git clone https://github.com/huggingface/fineweb-2.git -q

# ============================================================
# 🩹 Patch dependency check (avoid fasttext/glotlid hard error)
# ============================================================
import datatrove.utils._import_utils as iu
iu._raise_error_for_missing_dependencies = (
    lambda step, deps: print(f"[patch] skipping dependency check for {step}")
)

# ============================================================
# 🧾 Imports
# ============================================================
import json, os, glob
import pandas as pd
import matplotlib.pyplot as plt
import yaml

from datatrove.pipeline.readers.jsonl import JsonlReader
from datatrove.pipeline.writers.jsonl import JsonlWriter
from datatrove.pipeline.filters import (
    FineWebQualityFilter,
    GopherQualityFilter,
    GopherRepetitionFilter,
    LanguageFilter,
)
from datatrove.pipeline.dedup import (
    MinhashDedupSignature,
    MinhashDedupBuckets,
    MinhashDedupCluster,
    MinhashDedupFilter,
)
from datatrove.pipeline.dedup.minhash import MinhashConfig
from datatrove.pipeline.formatters import FTFYFormatter, PIIFormatter, SymbolLinesFormatter
from datatrove.utils.hashing import HashConfig

# ============================================================
# ⚙️ Load FineWeb2 official config (if exists)
# ============================================================
FW2_REPO_DIR = "/content/fineweb-2"
CONFIG_DIR = os.path.join(FW2_REPO_DIR, "configs")

lang_script = "eng_Latn"  

config_path = os.path.join(CONFIG_DIR, f"{lang_script}.yml")
if os.path.exists(config_path):
    with open(config_path, "r", encoding="utf-8") as f:
        filter_config = yaml.safe_load(f)
    print(f"✅ Loaded FineWeb2 config: {config_path}")
else:
    # English is not included in the multilingual configs, so here's a reasonable default version following the official FineWeb/FineWeb2 syntax.
    print(f"⚠️ Config {config_path} not found, using fallback defaults for {lang_script}")
    filter_config = {
        "language_score": 0.65,
        "dup_line_frac": 0.1,
        "top_n_grams": 100,
        "dup_n_grams": 20,
        "line_punct_thr": 0.4,
        "new_line_ratio": 0.4,
        "max_avg_word_length": 15,
        "min_avg_word_length": 2,
        "stopwords": [
            "the", "is", "in", "and", "to", "of", "that",
            "for", "on", "with", "as", "by", "at", "from"
        ],
        "max_non_alpha_words_ratio": 0.5,
    }
# Normalize top_n_grams / dup_n_grams to conform to the expected type of GopherRepetitionFilter.
tn = filter_config.get("top_n_grams")
dn = filter_config.get("dup_n_grams")

# If it's a single number, use the default configuration of the official Common Crawl Pipeline Creator.
if isinstance(tn, (int, float)):
    filter_config["top_n_grams"] = [(2, 0.2), (3, 0.18), (4, 0.16)]

if isinstance(dn, (int, float)):
    filter_config["dup_n_grams"] = [
        (5, 0.15),
        (6, 0.14),
        (7, 0.13),
        (8, 0.12),
        (9, 0.11),
        (10, 0.10),
    ]

# ============================================================
# 🚀 STEP 0: Load input data (with watermark)
# ============================================================
data_path = "/content/final_uniform_replace.jsonl"
records = [json.loads(line) for line in open(data_path, "r", encoding="utf-8")]

df = pd.DataFrame(records)
df = df[df.get("is_watermarked", False) == True].copy()
texts = df["watermarked"].astype(str).tolist()
print(f"✅ Loaded {len(texts)} watermarked documents for cleaning.")

input_jsonl = "/content/input_texts.jsonl"
with open(input_jsonl, "w", encoding="utf-8") as f:
    for i, t in enumerate(texts):
        f.write(json.dumps({"text": t, "id": i}) + "\n")

# ============================================================
# 🚀 STEP 1: Language Filter (Standalone LanguageFilter)

# - The official pipeline uses GlotLID + YAML's language_score for filtering in this step.

# - We don't have GlotLID to predict results, so we use ft176 + threshold to simulate.
# ============================================================
lang_threshold = filter_config.get("language_score", 0.65)

lang_filter = LanguageFilter(
    backend="ft176",
    language_threshold=lang_threshold,
    label_only=False,
)

reason_map = {}
lang_out_path = "/content/lang_filtered"
writer = JsonlWriter(lang_out_path)
passed_count = 0

for doc in JsonlReader(input_jsonl, text_key="text", id_key="id")():
    if lang_filter.filter(doc):
        writer.write(doc)
        passed_count += 1
    else:
        reason_map[doc.id] = "language_filter"
writer.close()

n_lang_pass = passed_count
print(f"✅ STEP1 (Language Filter): {passed_count}/{len(texts)} passed")
print(f"❌ Removed: {len(texts) - passed_count}")

# ============================================================
# 🧬 STEP 2: Minhash deduplication (align with the official 4-stage configuration)
# ============================================================
dedup_base = "/content/minhash"
os.makedirs(f"{dedup_base}/signatures", exist_ok=True)
os.makedirs(f"{dedup_base}/buckets", exist_ok=True)
os.makedirs(f"{dedup_base}/remove_ids", exist_ok=True)
os.makedirs(f"{dedup_base}/removed", exist_ok=True)

# —— Fully reuse the parameters of MinhashConfig in fineweb-2-pipeline.py —— #
minhash_config = MinhashConfig(
    hash_config=HashConfig(
        hash_fc="xxhash",
        precision=64,  # better precision -> fewer false positives
    ),
    num_buckets=14,
    hashes_per_bucket=8,
    n_grams=5,
)

# stage 1: compute signatures
MinhashDedupSignature(
    output_folder=f"{dedup_base}/signatures",
    config=minhash_config,
    language=lang_script, 
).run(JsonlReader(lang_out_path, text_key="text", id_key="id")())

# Stage 2: Bucketization — Simulating world_size = num_buckets number of workers
world_size = minhash_config.num_buckets  # 14 here
for rank in range(world_size):
    MinhashDedupBuckets(
        input_folder=f"{dedup_base}/signatures",
        output_folder=f"{dedup_base}/buckets",
        config=MinhashConfig(hash_config=minhash_config.hash_config),
    ).run(rank=rank, world_size=world_size)


# stage 3: clustering
MinhashDedupCluster(
    input_folder=f"{dedup_base}/buckets",
    output_folder=f"{dedup_base}/remove_ids",
    config=minhash_config,
    save_cluster_size=True,  
).run(world_size=1)

# stage 4: filter duplicates
dedup_output = f"{dedup_base}/output.jsonl"
dedup_removed_dir = f"{dedup_base}/removed"
dedup_removed_file = os.path.join(dedup_removed_dir, "removed.jsonl")

writer = JsonlWriter(dedup_output)
filt = MinhashDedupFilter(
    input_folder=f"{dedup_base}/remove_ids",
    exclusion_writer=JsonlWriter(dedup_removed_file),  
)

for doc in filt.run(JsonlReader(lang_out_path, text_key="text", id_key="id")()):
    writer.write(doc)
writer.close()

# ============================================================
# 🧩 Check deduplication results
# ============================================================
dedup_removed_ids = set()
removed_files = glob.glob(os.path.join(dedup_removed_dir, "*.jsonl*"))

for path in removed_files:
    for doc in JsonlReader(path, text_key="text", id_key="id")():
        dedup_removed_ids.add(doc.id)
        reason_map[doc.id] = "dedup"

print(f"✅ STEP2 (Minhash dedup): removed {len(dedup_removed_ids)} duplicates")

# ============================================================
# 🧹 STEP 3: Quality filtering + text fixing
# ============================================================

# 1) GopherRepetitionFilter
gopher_rep = GopherRepetitionFilter(
    language=lang_script,
    dup_para_frac=0,
    dup_line_char_frac=0,
    dup_para_char_frac=0,
    dup_line_frac=filter_config["dup_line_frac"],
    top_n_grams=filter_config["top_n_grams"],
    dup_n_grams=filter_config["dup_n_grams"],
)

# 2) FineWebQualityFilter
fw_quality = FineWebQualityFilter(
    language=lang_script,
    short_line_thr=999,
    char_duplicates_ratio=0.1,
    line_punct_thr=filter_config["line_punct_thr"],
    new_line_ratio=filter_config["new_line_ratio"],
)

# 3) GopherQualityFilter
gopher_qual = GopherQualityFilter(
    language=lang_script,
    max_avg_word_length=filter_config["max_avg_word_length"],
    min_avg_word_length=filter_config["min_avg_word_length"],
    stop_words=filter_config["stopwords"],
    max_non_alpha_words_ratio=filter_config["max_non_alpha_words_ratio"],
    min_stop_words=2,
)

# 4) FTFY + PII + fix table Format
ftfy_fmt = FTFYFormatter()
pii_fmt = PIIFormatter()
sym_fmt = SymbolLinesFormatter(symbols_to_remove=["|"], replace_char="\n")

cleaned_with_id = []
debug_log = []

for doc in JsonlReader(dedup_output, text_key="text", id_key="id")():
    if not hasattr(doc, "text") or not hasattr(doc, "id"):
        reason_map[getattr(doc, "id", -1)] = "missing_field"
        continue

    tid, text = doc.id, doc.text

    # gopher repetition
    if not gopher_rep.filter(doc):
        reason_map[tid] = "gopher_rep"
        continue

    # fineweb quality
    if not fw_quality.filter(doc):
        reason_map[tid] = "fw_quality"
        continue

    # gopher quality
    if not gopher_qual.filter(doc):
        reason_map[tid] = "gopher_qual"
        continue

    # final touches: FTFY -> PII -> SymbolLines
    text = sym_fmt.format(pii_fmt.format(ftfy_fmt.format(text)))
    cleaned_with_id.append((tid, text))

cleaned_dict = dict(cleaned_with_id)

# ============================================================
# 💧 STEP 4: 1-to-1 Invisible Watermark Detection
# ============================================================
mytext_path = "/content/myText.txt"
with open(mytext_path, "r", encoding="utf-8") as f:
    chars = f.read()

INVISIBLE_CHARS = [c for c in chars if not c.isprintable() and c != "\n"]
INVISIBLE_CODES = [f"U+{ord(c):04X}" for c in INVISIBLE_CHARS]
print(f"💧 Loaded {len(INVISIBLE_CHARS)} invisible watermark characters:")
print(", ".join(INVISIBLE_CODES[:10]) + (" ..." if len(INVISIBLE_CODES) > 10 else ""))

def count_invisible(text: str, idx: int) -> int:
    if idx >= len(INVISIBLE_CHARS):
        return 0
    return text.count(INVISIBLE_CHARS[idx])

def has_invisible(text: str, idx: int) -> bool:
    return count_invisible(text, idx) > 0

orig_with_wm = sum(has_invisible(t, i) for i, t in enumerate(texts))
clean_with_wm = sum(has_invisible(cleaned_dict.get(i, ""), i) for i in range(len(texts)))
doc_retention_rate = (clean_with_wm / orig_with_wm * 100) if orig_with_wm else 0.0

removed_doc_ids = [i for i in range(len(texts)) if i not in cleaned_dict]

print(f"\n💧 Original docs with watermark: {orig_with_wm}")
print(f"💧 Cleaned docs with watermark:  {clean_with_wm}")
print(f"✅ Document-level retention rate: {doc_retention_rate:.2f}%")

# ============================================================
# 📁 STEP 5: Output removed documents + removal reasons
# ============================================================
rows = []
for rid in removed_doc_ids:
    rows.append({
        "doc_id": rid,
        "reason": reason_map.get(rid, "unknown"),
        "raw_text": texts[rid][:400].replace("\n", " "),
        "zwc_count": count_invisible(texts[rid], rid),
        "orig_len": len(texts[rid]),
    })
df_removed = pd.DataFrame(rows)
print("\n📊 Removal summary by reason:")
print(df_removed["reason"].value_counts())

removed_csv = "/content/removed_docs_with_text.csv"
df_removed.to_csv(removed_csv, index=False)
print(f"\n📁 Full removed docs saved to {removed_csv}")

# ============================================================
# 📈 STEP 6: Visualize document counts by stage
# ============================================================
stage_counts = {
    "original": len(texts),
    "lang_filter": n_lang_pass,
    "dedup": n_lang_pass - len(dedup_removed_ids),
    "quality": len(cleaned_with_id),
}

print("\n📈 Stage counts summary:")
for stage, count in stage_counts.items():
    print(f"  {stage:<15}: {count}")

plt.figure(figsize=(6, 4))
plt.bar(stage_counts.keys(), stage_counts.values())  
plt.ylabel("Remaining documents")
plt.title("Document count after each FineWeb2-like cleaning stage")
plt.grid(alpha=0.3)
plt.show()
